# ACELO Cluster Approval Tracking

Maintains the `cluster_optimization_tracking` Delta table that ACELO's in-app
**Approvals** tab reads directly from OneLake.

This is the tracking logic of the retired "cluster email fabric" notebook with
the email removed entirely: no mail server, no credentials, no email
columns. Approval decisions are made and recorded in ACELO, not here.

**Managed by ACELO.** Runs after the Cluster notebook in the ACELO pipeline.
Read-only against the optimizer's result table; writes only the tracking table.


In [ ]:
# PARAMETERS CELL - values are injected by the ACELO pipeline at run time.
acelo_run_id = ""        # this run's rows only; blank = the latest run in the result table
result_table = ""        # the Cluster optimizer's result table (read)
result_schema = ""       # e.g. "dbo" for a schema-enabled Lakehouse
approval_tracking_table = ""   # the tracking table (written)


In [ ]:
import re

_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]{0,127}$")


def _name(table, schema):
    table, schema = (table or "").strip(), (schema or "").strip()
    for part in [p for p in (schema, table) if p]:
        if not _IDENT.match(part):
            raise ValueError(f"Invalid table/schema name: {part!r}")
    return f"{schema}.{table}" if schema else table


_missing = [n for n, v in (("result_table", result_table),
                           ("approval_tracking_table", approval_tracking_table)) if not (v or "").strip()]
if _missing:
    raise ValueError("ACELO approval tracking is missing required parameters: " + ", ".join(_missing))

SOURCE = _name(result_table, result_schema)
TRACKING = _name(approval_tracking_table, result_schema)
RUN_ID = (acelo_run_id or "").strip()
if RUN_ID and not re.match(r"^[A-Za-z0-9_\-]{1,128}$", RUN_ID):
    raise ValueError("Invalid acelo_run_id")

print("[APPROVAL_TRACKING]")
print(f"source={SOURCE}")
print(f"tracking={TRACKING}")
print(f"acelo_run_id={RUN_ID or '(latest)'}")

# Same table the legacy notebook created, minus its email-only columns. On a
# table that already exists (with the legacy email columns) this is a no-op.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TRACKING} (
    cluster_name STRING,
    optimization_label STRING,
    current_workers INT,
    recommended_max_workers INT,
    total_dbus_cost_usd DOUBLE,
    potential_monthly_savings DOUBLE,
    llm_optimization STRING,
    status STRING,
    updated_at TIMESTAMP,
    acelo_run_id STRING
)
USING DELTA
""")

# Additive only: older tracking tables have no acelo_run_id column.
if "acelo_run_id" not in spark.table(TRACKING).columns:
    spark.sql(f"ALTER TABLE {TRACKING} ADD COLUMNS (acelo_run_id STRING)")

# The result table is append-only (one row set per run), so select ONE run's
# rows; otherwise the same cluster would reach the MERGE more than once.
if not RUN_ID:
    _latest = spark.sql(f"""
        SELECT acelo_run_id FROM {SOURCE}
        WHERE acelo_run_id IS NOT NULL
        ORDER BY acelo_analyzed_at DESC LIMIT 1
    """).collect()
    if not _latest:
        raise ValueError(f"No ACELO runs found in {SOURCE}; nothing to track.")
    RUN_ID = _latest[0]["acelo_run_id"]
    print(f"acelo_run_id resolved to latest: {RUN_ID}")

# Legacy business rule, unchanged: Risky / Moderately Optimized clusters with a
# name become PENDING approval candidates. Existing rows are left untouched -
# their approval state is owned by ACELO now.
spark.sql(f"""
MERGE INTO {TRACKING} t
USING (
    SELECT
        cluster_name,
        optimization_label,
        CAST(current_workers AS INT)          AS current_workers,
        CAST(recommended_max_workers AS INT)  AS recommended_max_workers,
        CAST(total_dbus_cost_usd AS DOUBLE)   AS total_dbus_cost_usd,
        CAST(potential_monthly_savings AS DOUBLE) AS potential_monthly_savings,
        llm_optimization,
        acelo_run_id
    FROM {SOURCE}
    WHERE acelo_run_id = '{RUN_ID}'
      AND optimization_label IN ('Risky', 'Moderately Optimized')
      AND cluster_name IS NOT NULL
      AND TRIM(cluster_name) <> ''
) s
ON t.cluster_name = s.cluster_name
WHEN NOT MATCHED THEN
INSERT (
    cluster_name, optimization_label, current_workers, recommended_max_workers,
    total_dbus_cost_usd, potential_monthly_savings, llm_optimization,
    status, updated_at, acelo_run_id
)
VALUES (
    s.cluster_name, s.optimization_label, s.current_workers, s.recommended_max_workers,
    s.total_dbus_cost_usd, s.potential_monthly_savings, s.llm_optimization,
    'PENDING', current_timestamp(), s.acelo_run_id
)
""")

_pending = spark.sql(f"SELECT COUNT(*) AS n FROM {TRACKING} WHERE status = 'PENDING'").collect()[0]["n"]
print("[APPROVAL_TRACKING]")
print("STATUS=SUCCESS")
print(f"pending_candidates={_pending}")
